# Этап 5: сравнение IrResnet4 (hidden=72)

E1–E4: SMARTS-only vs SMARTS+peak × контекст измерения on/off.
Требуется `dataset_v002` с `labels_structure_smarts.parquet`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"  # при необходимости замените
# Локально: cwd уже корень репо; Colab: клонируем в IR_expert_system_3
ROOT = Path.cwd()
if (ROOT / 'pyproject.toml').is_file():
    pass
elif (ROOT / 'IR_expert_system_3' / 'pyproject.toml').is_file():
    ROOT = ROOT / 'IR_expert_system_3'
elif Path('/content/IR_expert_system_3/pyproject.toml').is_file():
    ROOT = Path('/content/IR_expert_system_3')
else:
    REPO_DIR = Path('IR_expert_system_3')
    if not REPO_DIR.is_dir():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    ROOT = REPO_DIR.resolve()
import os
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
print('ROOT:', ROOT.resolve())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'], check=True)
!ir-pipeline --help
help_txt = subprocess.check_output(['ir-pipeline', '--help'], text=True)
if ' run ' not in help_txt:
    print('WARNING: команда `run` отсутствует. Ноутбук использует fallback без run-stage.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
IR_DATA = Path('/content/drive/MyDrive/ir_data')
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)


## Датасет: загрузка вручную

1. **Files → Upload** в Colab: `dataset_v001.zip` / `dataset_mini.zip` в `/content` (или положите архив на Google Drive).
2. Выполните ячейку распаковки ниже — ожидается `data/processed/<версия>/spectra.npz`.
3. Если архива нет — следующая ячейка скачает мини-датасет с Hugging Face.


In [ ]:
from pathlib import Path
import zipfile

DATASET_VERSIONS = ('dataset_mini', 'dataset_v001')
SEARCH_ROOTS = [
    Path('/content'),
    Path('/content/IR_expert_system_3'),
    Path('/content/drive/MyDrive'),
    Path('/content/drive/MyDrive/ir_data'),
    Path('.'),
]
try:
    SEARCH_ROOTS.insert(0, IR_DATA)
except NameError:
    pass
DEST = Path('data/processed')
DEST.mkdir(parents=True, exist_ok=True)

def _dataset_ready(name: str) -> bool:
    return (DEST / name / 'spectra.npz').is_file()

def _find_zip_archives() -> list[Path]:
    found: list[Path] = []
    seen: set[str] = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob('*.zip'):
            key = str(p.resolve())
            if key in seen:
                continue
            low = p.name.lower()
            if any(v in low for v in DATASET_VERSIONS):
                seen.add(key)
                found.append(p)
    return sorted(found, key=lambda x: x.stat().st_mtime, reverse=True)

archives = _find_zip_archives()
print('Найденные zip с датасетом:')
if archives:
    for p in archives[:15]:
        print(f'  {p} ({p.stat().st_size / 1e6:.1f} MB)')
else:
    print('  (нет — загрузите через Files → Upload)')

for version in DATASET_VERSIONS:
    if _dataset_ready(version):
        print(f'OK: {DEST / version} уже распакован')
        continue
    matched = [p for p in archives if version in p.name.lower()]
    if not matched:
        print(f'Пропуск {version}: zip не найден')
        continue
    zp = matched[0]
    print(f'Распаковка {zp.name} → {DEST}')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(DEST)
    if _dataset_ready(version):
        print(f'  → готово: {DEST / version / "spectra.npz"}')
    else:
        print(
            f'  WARNING: после распаковки нет {DEST / version / "spectra.npz"}. '
            'Проверьте структуру zip (внутри должна быть папка {version}/).'
        )


In [ ]:
from pathlib import Path

DATASET_DIR = Path('data/processed/dataset_mini')
if DATASET_DIR.joinpath('spectra.npz').is_file():
    print(f'OK: {DATASET_DIR}')
else:
    print('dataset_mini not found → fetching from HF...')
    !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed


In [ ]:
%matplotlib inline


In [ ]:
from pathlib import Path
import json
import pandas as pd
from ir_pipeline.config_loader import load_yaml, merge_train_defaults, resolve_paths
from ir_pipeline.dataset_preview import build_multilabel_matrix
from ir_pipeline.resnet_input import load_model_inputs

paths = resolve_paths(load_yaml(Path('configs/paths.huggingface.yaml')))
DATASET = paths['processed_root'] / 'dataset_v002'
_, _, spec_ids, _, _ = load_model_inputs(DATASET)
bands = paths['bands_config']
rows = []
for schema in ['structure_smarts', 'structure', 'spectrum']:
    try:
        Y, _ = build_multilabel_matrix(DATASET, spec_ids, bands, label_schema=schema)
        rows.append({'schema': schema, 'mean_labels': float(Y.sum(axis=1).mean()), 'positives': int(Y.sum())})
    except FileNotFoundError as e:
        rows.append({'schema': schema, 'error': str(e)})
pd.DataFrame(rows)


In [ ]:
from copy import deepcopy
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import train_irresnet_run

base_cfg = merge_train_defaults(load_yaml(Path('configs/train_irresnet_experiments.yaml')))
EXPERIMENTS = [
    ('E1', 'structure_smarts', False),
    ('E2', 'structure', False),
    ('E3', 'structure_smarts', True),
    ('E4', 'structure', True),
]
summaries = []
for exp_id, schema, use_ctx in EXPERIMENTS:
    cfg = deepcopy(base_cfg)
    cfg['use_measurement_context'] = use_ctx
    run_dir = Path('runs/exp_v002') / f'{exp_id.lower()}_h72_{"ctx" if use_ctx else "noctx"}_{schema}'
    print('===', exp_id, schema, 'context=', use_ctx, '=>', run_dir)
    s = train_irresnet_run(
        dataset_dir=DATASET,
        run_dir=run_dir,
        bands_yaml=bands,
        train_cfg=cfg,
        label_schema=schema,
        use_measurement_context=use_ctx,
    )
    s['experiment'] = exp_id
    summaries.append(s)
pd.DataFrame(summaries)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

df = pd.DataFrame(summaries)
out = Path('runs/exp_v002')
out.mkdir(parents=True, exist_ok=True)
(out / 'summary.json').write_text(df.to_json(orient='records', indent=2), encoding='utf-8')
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(df))
ax.bar(x, df['test_f1_weighted'], color='steelblue')
ax.set_xticks(list(x))
ax.set_xticklabels(df['experiment'], rotation=0)
ax.set_ylabel('test F1 weighted')
ax.set_title('IrResnet4 experiments (hidden=72)')
fig.tight_layout()
fig.savefig(out / 'experiments_f1_weighted.png', dpi=140)
plt.show()
df
